In [11]:
import torch
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.image import resize
import matplotlib.pyplot as plt
import os

# === CONFIGURACIÓN ===

# Ruta al modelo YOLOv5 (usa 'yolov5s.pt' por defecto, entrenado en COCO)
yolo_model = torch.hub.load('yolov5', 'yolov5s', source='local')  # usa repositorio clonado

# Ruta al modelo entrenado .h5
cnn_model = load_model('supermarket_model.h5')

# Etiquetas de tu modelo .h5 (en orden exacto)
etiquetas = [
    'Banana', 'Orange', 'Red-Bell-Pepper', 'aluminium_form', 'bread',
    'doritos_cocaCola', 'red_bull', 'shampoo_H&S', 'tea', 'yogurt_toni_mix'
]

# Tamaño de entrada que espera tu CNN
IMAGE_SIZE = (224, 224)  # ajusta si tu modelo usa otro tamaño

# Umbral de confianza de detección YOLO
conf_threshold = 0.3

# === FUNCIÓN PRINCIPAL ===

def detectar_y_clasificar(imagen_path):
    # Leer imagen
    img = cv2.imread(imagen_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # === Paso 1: Detección con YOLOv5 ===
    detecciones = yolo_model(img_rgb)
    resultados = detecciones.pandas().xyxy[0]  # DataFrame con [xmin, ymin, xmax, ymax, confidence, class, name]

    objetos_detectados = []

    for i, row in resultados.iterrows():
        if row['confidence'] >= conf_threshold:
            x1, y1, x2, y2 = map(int, [row['xmin'], row['ymin'], row['xmax'], row['ymax']])
            crop = img_rgb[y1:y2, x1:x2]

            if crop.shape[0] == 0 or crop.shape[1] == 0:
                continue  # saltar regiones vacías

            # === Paso 2: Clasificación con tu modelo CNN ===
            resized = cv2.resize(crop, IMAGE_SIZE)
            array = img_to_array(resized) / 255.0
            pred = cnn_model.predict(np.expand_dims(array, axis=0))[0]

            idx = np.argmax(pred)
            confidence = pred[idx]
            label = etiquetas[idx]

            objetos_detectados.append({
                "label": label,
                "conf": float(confidence),
                "bbox": (x1, y1, x2, y2)
            })

    # === Visualización ===
    for obj in objetos_detectados:
        x1, y1, x2, y2 = obj["bbox"]
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        text = f"{obj['label']} ({obj['conf']:.2f})"
        cv2.putText(img_rgb, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title("Objetos detectados y clasificados")
    plt.show()

    # === Consola ===
    print("Productos detectados en la imagen:")
    for i, obj in enumerate(objetos_detectados):
        print(f"{i+1}. {obj['label']} - {obj['conf']:.2f}")

# === USO ===
imagen_prueba = 'ruta/a/tu/imagen.jpg'  # <-- cambia por tu ruta
detectar_y_clasificar(imagen_prueba)


ModuleNotFoundError: No module named 'torch'